In [ ]:
import glob
import pandas as pd, numpy as np, folium
from saajha.clean import haversine_km, remove_jumps

f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = pd.read_csv(f).drop_duplicates(["vehicle_id", "timestamp"]).dropna(subset=["route_id"])
clean = pd.concat([remove_jumps(g) for _, g in pos.groupby("vehicle_id")])

In [ ]:
CELL = 0.0015  # about 150 m squares
g = clean.groupby("vehicle_id")
span_km = haversine_km(g.lat.min(), g.lon.min(), g.lat.max(), g.lon.max())
keep = span_km[(span_km > 2) & (g.size() >= 20)].index
tr = clean[clean.vehicle_id.isin(keep)].copy()

tr["cell"] = list(zip((tr.lat / CELL).round().astype(int), (tr.lon / CELL).round().astype(int)))
cells = tr.groupby("vehicle_id")["cell"].apply(set)
true_route = tr.groupby("vehicle_id")["route_id"].agg(lambda s: s.mode().iat[0]).loc[cells.index]
print(len(cells), "moving buses,", true_route.nunique(), "true routes")

In [ ]:
from scipy.sparse import csr_matrix
ids = cells.index
cell_id = {c: i for i, c in enumerate(set().union(*cells))}
rows = [r for r, s in enumerate(cells) for _ in s]
cols = [cell_id[c] for s in cells for c in s]
M = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(ids), len(cell_id)))

inter = (M @ M.T).toarray()
size = np.asarray(M.sum(axis=1)).ravel()
dist = 1 - inter / np.minimum.outer(size, size)

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import homogeneity_score, completeness_score

pred = DBSCAN(eps=0.3, min_samples=2, metric="precomputed").fit_predict(dist)
mask = pred >= 0
print(f"{pred.max() + 1} clusters | {mask.mean():.0%} of buses clustered")
print(f"homogeneity {homogeneity_score(true_route[mask], pred[mask]):.2f} | "
      f"completeness {completeness_score(true_route[mask], pred[mask]):.2f}")

In [ ]:
colors = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue", "darkgreen"]
top = pd.Series(pred[mask]).value_counts().index[:8]
m4 = folium.Map(location=[28.61, 77.21], zoom_start=11, tiles="OpenStreetMap")
for color, c in zip(colors, top):
    for vid in ids[pred == c]:
        t = tr[tr.vehicle_id == vid].sort_values("timestamp")
        folium.PolyLine(t[["lat", "lon"]].values.tolist(), color=color, weight=3, opacity=0.6).add_to(m4)
m4.save("../data/processed/routes_map.html")
m4

In [ ]:
from sklearn.cluster import AgglomerativeClustering

for t in [0.3, 0.5, 0.7]:
    lab = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                                  linkage="average", distance_threshold=t).fit_predict(dist)
    sizes = pd.Series(lab).value_counts()
    m = pd.Series(lab).map(sizes).values >= 2
    print(f"t={t}: {(sizes >= 2).sum()} clusters | {m.mean():.0%} clustered | "
          f"homogeneity {homogeneity_score(true_route[m], lab[m]):.2f} | "
          f"completeness {completeness_score(true_route[m], lab[m]):.2f}")

In [ ]:
lab = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                              linkage="average", distance_threshold=0.5).fit_predict(dist)
top = pd.Series(lab).value_counts().index[:8]
m5 = folium.Map(location=[28.61, 77.21], zoom_start=11, tiles="OpenStreetMap")
for color, c in zip(colors, top):
    for vid in ids[lab == c]:
        t = tr[tr.vehicle_id == vid].sort_values("timestamp")
        folium.PolyLine(t[["lat", "lon"]].values.tolist(), color=color, weight=3, opacity=0.6).add_to(m5)
m5.save("../data/processed/routes_map_avg.html")
m5

In [ ]:
tr = tr.sort_values(["vehicle_id", "timestamp"])
nxt = tr.groupby("vehicle_id")[["lat", "lon"]].shift(-1)
lat1, lat2 = np.radians(tr.lat), np.radians(nxt.lat)
dlon = np.radians(nxt.lon - tr.lon)
x = np.sin(dlon) * np.cos(lat2)
y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
tr["bearing"] = (np.degrees(np.arctan2(x, y)) + 360) % 360
tr["step_km"] = haversine_km(tr.lat, tr.lon, nxt.lat, nxt.lon)

mv = tr[tr.bearing.notna() & (tr.step_km > 0.03)].copy()
mv["hbin"] = ((mv.bearing + 45) // 90 % 4).astype(int)   # 0=N, 1=E, 2=S, 3=W
mv["tok"] = list(zip(mv.cell, mv.hbin))
tokens = mv.groupby("vehicle_id")["tok"].apply(set)
print(len(tokens), "buses with direction tokens")

In [ ]:
def overlap_dist(sets):
    vocab = {c: i for i, c in enumerate(set().union(*sets))}
    rows = [r for r, s in enumerate(sets) for _ in s]
    cols = [vocab[c] for s in sets for c in s]
    M = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(vocab)))
    inter = (M @ M.T).toarray()
    size = np.asarray(M.sum(axis=1)).ravel()
    return 1 - inter / np.minimum.outer(size, size)

def evaluate(dist, truth, t):
    lab = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                                  linkage="average", distance_threshold=t).fit_predict(dist)
    sizes = pd.Series(lab).value_counts()
    m = pd.Series(lab).map(sizes).values >= 2
    return (sizes >= 2).sum(), m.mean(), homogeneity_score(truth[m], lab[m]), completeness_score(truth[m], lab[m])

truth = true_route.loc[tokens.index].values
for name, sets in [("no direction", cells.loc[tokens.index]), ("with direction", tokens)]:
    d = overlap_dist(list(sets))
    for t in [0.5, 0.7]:
        n, cov, h, c = evaluate(d, truth, t)
        print(f"{name:15s} t={t}: {n} clusters | {cov:.0%} clustered | homogeneity {h:.2f} | completeness {c:.2f}")

In [ ]:
import matplotlib.pyplot as plt

rows = []
for name, sets in [("no direction", cells.loc[tokens.index]), ("with direction", tokens)]:
    d = overlap_dist(list(sets))
    for t in np.arange(0.3, 0.86, 0.05):
        n, cov, h, c = evaluate(d, truth, t)
        rows.append((name, round(t, 2), cov, h, c))
res = pd.DataFrame(rows, columns=["method", "t", "coverage", "homogeneity", "completeness"])

for name, g in res.groupby("method"):
    plt.plot(g.coverage, g.homogeneity, marker="o", label=name)
plt.xlabel("coverage (share of buses clustered)")
plt.ylabel("homogeneity (route purity)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("../docs/direction_tradeoff.png", dpi=150)
plt.show()

In [ ]:
%pip install matplotlib